# Phase 2: Preprocessing & Feature Engineering
### AI-Based Energy Anomaly Detector
**Goal**: Convert noisy minute-level power signals into hourly engineered features with cyclical and baseline context.

---
### Rationale for Feature Engineering
1. **Hourly Resampling**: Minute data contains random transient spikes (e.g. motor starts). Hourly aggregation produces smooth signals suitable for practical alert systems.
2. **Linear Interpolation**: Heals gaps without distorting physical trends.
3. **Hour of Day (0–23)**: Diurnal pattern distinction (day vs. night baselines).
4. **Day of Week (0–6)**: Weekday vs. weekend behavioral shifts.
5. **24h Rolling Average**: Contextual baseline — helps the model detect anomalies relative to recent consumption rather than all-time averages.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is on sys.path to import src modules
proj_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(proj_dir / 'src') not in sys.path:
    sys.path.append(str(proj_dir / 'src'))

from data_loader import load_raw_data
from feature_engineering import build_hourly_features, save_processed_datasets

## 1. Load Raw Minute Data

In [ ]:
raw_csv_path = proj_dir / 'data' / 'raw' / 'household_power_consumption.txt'
df_raw = load_raw_data(raw_csv_path)
print(f"Raw dataset shape: {df_raw.shape}")

## 2. Resample & Engineer Features

In [ ]:
features, train_df, test_df = build_hourly_features(df_raw, target_col='Global_active_power', train_ratio=0.80)
print("\n--- Feature Table Summary ---")
features.info()
features.head(10)

## 3. Verify Absence of Missing Values

In [ ]:
print("Missing values across features:")
print(features.isna().sum())

## 4. Visual Comparison: Minute vs. Hourly Resampled Data

In [ ]:
sample_range = slice('2007-03-01', '2007-03-03')
plt.figure(figsize=(15, 6), dpi=120)

plt.plot(df_raw.loc[sample_range].index, df_raw.loc[sample_range, 'Global_active_power'], 
         color='lightgray', alpha=0.8, label='Minute-level raw active power')
plt.plot(features.loc[sample_range].index, features.loc[sample_range, 'usage'], 
         color='#2563eb', linewidth=2.0, label='Hourly aggregated mean (usage)')
plt.plot(features.loc[sample_range].index, features.loc[sample_range, 'rolling_mean_24h'], 
         color='#ea580c', linewidth=2.0, linestyle='--', label='24-Hour rolling baseline')

plt.title('Minute Raw Noise vs. Hourly Mean & 24h Rolling Baseline (March 1–3, 2007)', fontsize=13, fontweight='bold')
plt.xlabel('Timestamp')
plt.ylabel('Power (kW)')
plt.legend(frameon=True)
plt.tight_layout()

comp_plot_path = proj_dir / 'outputs' / 'plots' / 'minute_vs_hourly_comparison.png'
plt.savefig(comp_plot_path, dpi=300)
print(f"Comparison plot saved to: {comp_plot_path}")
plt.show()

## 5. Save Processed Datasets
Persist full feature table as well as train and test splits to `data/processed/`.

In [ ]:
save_processed_datasets(features, train_df, test_df, output_dir=proj_dir / 'data' / 'processed')
print("Preprocessing & feature engineering complete!")